## Modelling prechecks

**Objective:** Verify two assumptions flagged in `modelling_decisions.md` before feature extraction begins, so neither assumption is silently carried into the pipeline unchecked.

1. **Protocol composition vs IAF-proximity (gates decision 5).** IAF-proximity-to-10Hz replicated in Roelofs et al. (2021) only for the 10Hz unilateral left-DLPFC subgroup, not for 1Hz or bilateral protocols. This check establishes the protocol composition of the 163-subject cohort and whether protocol is itself confounded with responder status, to determine whether IAF-proximity can be used cohort-wide, needs restricting to a subgroup, or needs protocol as a covariate.

2. **Age vs retained-epoch-count (gates decision 3).** Age is an established confound on responder status (younger subjects respond more, Welch's t=-2.68, p=.01). The ~10-epoch inclusion floor was set on autoreject-stability grounds, not tested against age. This check establishes whether epoch retention correlates with age, which would mean the floor systematically excludes subjects by age and deepens rather than controls the confound.

**Inputs:**
- Check 1: `data/cohort_filtered_n163.xlsx` (full 163-subject cohort, protocol field, responder label)
- Check 2: QC metadata from the full 163-subject preprocessing run (`04_full_cohort_run.ipynb`)

**Assumptions / limitations stated upfront:**
- Check 1 runs on the full 163-subject cohort as defined by clinical/label criteria.
- Check 2 runs on the 160 subjects with usable QC metadata from the full preprocessing run
  (`04_full_cohort_run.ipynb`, logged in `data/batch_results_log_full_cohort.csv`). Three subjects
  in the 163-subject cohort have no raw BDF file on disk and are excluded here as a data
  availability gap, not a preprocessing failure, worth flagging if either excluded subject
  distribution turns out responder/non-responder imbalanced.
- Check 2 uses one row per subject (restEC condition, heog_off variant, the primary dataset
  variant per decision 1) to avoid double-counting across conditions/variants.

In [1]:
# imports and setup
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import importlib
import src.preprocessing
importlib.reload(src.preprocessing)
from src.preprocessing import *

In [2]:
#Cohort loading 
data_dir = find_repo_root() / "data"

cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")
all_subject_ids = cohort_df['TDBRAIN_ID'].tolist()
print(f"{len(all_subject_ids)} subjects in cohort")

163 subjects in cohort


In [3]:
# Check available columns before assuming a specific protocol-field name 
# TDBRAIN's spreadsheet may label this stim frequency/hemisphere field
# differently than expected (e.g. 'stim_protocol', 'location', 'frequency')
print(cohort_df.columns.tolist())

['TDBRAIN_ID', 'DISC/REP', 'indication', 'formal_status', 'Dataset', 'Consent', 'sessSeason', 'sessTime', 'Responder', 'Remitter', 'age', 'gender', 'sessID', 'nrSessions', 'neoFFI_q1', 'neoFFI_q2', 'neoFFI_q3', 'neoFFI_q4', 'neoFFI_q5', 'neoFFI_q6', 'neoFFI_q7', 'neoFFI_q8', 'neoFFI_q9', 'neoFFI_q10', 'neoFFI_q11', 'neoFFI_q12', 'neoFFI_q13', 'neoFFI_q14', 'neoFFI_q15', 'neoFFI_q16', 'neoFFI_q17', 'neoFFI_q18', 'neoFFI_q19', 'neoFFI_q20', 'neoFFI_q21', 'neoFFI_q22', 'neoFFI_q23', 'neoFFI_q24', 'neoFFI_q25', 'neoFFI_q26', 'neoFFI_q27', 'neoFFI_q28', 'neoFFI_q29', 'neoFFI_q30', 'neoFFI_q31', 'neoFFI_q32', 'neoFFI_q33', 'neoFFI_q34', 'neoFFI_q35', 'neoFFI_q36', 'neoFFI_q37', 'neoFFI_q38', 'neoFFI_q39', 'neoFFI_q40', 'neoFFI_q41', 'neoFFI_q42', 'neoFFI_q43', 'neoFFI_q44', 'neoFFI_q45', 'neoFFI_q46', 'neoFFI_q47', 'neoFFI_q48', 'neoFFI_q49', 'neoFFI_q50', 'neoFFI_q51', 'neoFFI_q52', 'neoFFI_q53', 'neoFFI_q54', 'neoFFI_q55', 'neoFFI_q56', 'neoFFI_q57', 'neoFFI_q58', 'neoFFI_q59', 'neoFFI_q60

In [5]:
# Inspect raw values before crosstabbing - clinical spreadsheet fields often have
# inconsistent formatting (spacing, capitalisation) or missing values that would
# silently fragment the crosstab into spurious extra categories if not caught first
print(cohort_df['rTMS PROTOCOL'].value_counts(dropna=False))

rTMS PROTOCOL
2.0    85
1.0    42
3.0    32
NaN     4
Name: count, dtype: int64


In [6]:
# Check whether the 7-vs-32 discrepancy against van Dijk et al.'s Table 2 is a
# denominator artefact (their "7" was counted under a stricter BDI pre&post
# filter) or a genuine dataset-version difference between the 2022 paper and
# our V3.1 release. Filtering to subjects with both BDI scores present should
# reproduce something close to Table 2's counts (65 / 105 / 7) if it's just
# a denominator issue - if the gap persists, it's a real V3.1 vs. paper difference.
bdi_complete = cohort_df[cohort_df['BDI_pre'].notna() & cohort_df['BDI_post'].notna()]

print(f"{len(bdi_complete)} subjects with complete BDI pre/post (of {len(cohort_df)} total)")
print("\nProtocol counts, BDI-complete subjects only:")
print(bdi_complete['rTMS PROTOCOL'].value_counts(dropna=False))

print("\nProtocol counts, full 163-subject cohort (for comparison):")
print(cohort_df['rTMS PROTOCOL'].value_counts(dropna=False))

163 subjects with complete BDI pre/post (of 163 total)

Protocol counts, BDI-complete subjects only:
rTMS PROTOCOL
2.0    85
1.0    42
3.0    32
NaN     4
Name: count, dtype: int64

Protocol counts, full 163-subject cohort (for comparison):
rTMS PROTOCOL
2.0    85
1.0    42
3.0    32
NaN     4
Name: count, dtype: int64


In [7]:
# Check whether responder status differs by protocol - if response rates diverge
# by protocol, that's a second confound sitting alongside age (Decision 7), and
# it matters here specifically because protocol is already being used to gate
# which subjects get the IAF-proximity feature (Decision 5). A crosstab with
# normalized rows shows response rate per protocol directly.
protocol_response = pd.crosstab(cohort_df['rTMS PROTOCOL'], cohort_df['Responder'])
protocol_response_pct = pd.crosstab(cohort_df['rTMS PROTOCOL'], cohort_df['Responder'], normalize='index')

print("Counts:")
print(protocol_response)
print("\nResponse rate by protocol:")
print(protocol_response_pct)

# Chi-square test for independence: is responder status associated with protocol?
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(protocol_response.dropna())
print(f"\nChi-square test: chi2={chi2:.2f}, p={p:.3f}")

Counts:
Responder       0   1
rTMS PROTOCOL        
1.0            17  25
2.0            31  54
3.0            21  11

Response rate by protocol:
Responder             0         1
rTMS PROTOCOL                    
1.0            0.404762  0.595238
2.0            0.364706  0.635294
3.0            0.656250  0.343750

Chi-square test: chi2=8.24, p=0.016
